In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

from src.data.generate import generate
from src.data.preprocess import preprocess
from src.utils.config import FEATURES, TARGET

raw_df = generate(n=3000, seed=42, save=False)
df     = preprocess(raw_df, save=False)

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'No-show rate in test: {y_test.mean():.2%}')

Train size: 2400 | Test size: 600
No-show rate in test: 23.00%


In [2]:
def evaluate(name, model, X_test, y_test, threshold=0.5):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm  = confusion_matrix(y_test, y_pred)

    print(f'\n{"-"*50}')
    print(f'  EXPERIMENT: {name}')
    print(f'  Threshold used: {threshold}')
    print(f'{"-"*50}')
    print(f'  Accuracy:      {acc:.4f}')
    print(f'  ROC-AUC:       {auc:.4f}')
    print(f'  F1 (No-Show):  {f1:.4f}')
    print(f'  Confusion Matrix:')
    print(f'    TN={cm[0][0]}  FP={cm[0][1]}')
    print(f'    FN={cm[1][0]}  TP={cm[1][1]}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Show", "No-Show"])}')

    return {'name': name, 'accuracy': acc, 'f1': f1, 'roc_auc': auc, 'threshold': threshold}

results = []
print('Helper ready!')

Helper ready!


## Experiment 1 Baseline (class_weight=balanced, threshold=0.5)

In [3]:
model_v1 = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model_v1.fit(X_train, y_train)

r1 = evaluate('v1 - balanced, threshold=0.5', model_v1, X_test, y_test, threshold=0.5)
results.append(r1)


--------------------------------------------------
  EXPERIMENT: v1 - balanced, threshold=0.5
  Threshold used: 0.5
--------------------------------------------------
  Accuracy:      0.7617
  ROC-AUC:       0.5735
  F1 (No-Show):  0.0892
  Confusion Matrix:
    TN=450  FP=12
    FN=131  TP=7

              precision    recall  f1-score   support

        Show       0.77      0.97      0.86       462
     No-Show       0.37      0.05      0.09       138

    accuracy                           0.76       600
   macro avg       0.57      0.51      0.48       600
weighted avg       0.68      0.76      0.68       600



# Experiment 2  Lower Threshold to 0.30
Same model as v1, but we lower the decision bar for predicting no-show.

In [4]:
r2 = evaluate('v2 - balanced, threshold=0.30', model_v1, X_test, y_test, threshold=0.30)
results.append(r2)


--------------------------------------------------
  EXPERIMENT: v2 - balanced, threshold=0.30
  Threshold used: 0.3
--------------------------------------------------
  Accuracy:      0.6283
  ROC-AUC:       0.5735
  F1 (No-Show):  0.3180
  Confusion Matrix:
    TN=325  FP=137
    FN=86  TP=52

              precision    recall  f1-score   support

        Show       0.79      0.70      0.74       462
     No-Show       0.28      0.38      0.32       138

    accuracy                           0.63       600
   macro avg       0.53      0.54      0.53       600
weighted avg       0.67      0.63      0.65       600



## Experiment 3 Stronger Class Weight {0:1, 1:4}
We punish the model = missing a no-show is 4x worse than missing a show.

In [5]:
model_v3 = RandomForestClassifier(
    n_estimators=300,
    class_weight={0: 1, 1: 4},
    random_state=42,
    n_jobs=-1
)
model_v3.fit(X_train, y_train)

r3 = evaluate('v3 - weight {0:1,1:4}, threshold=0.30', model_v3, X_test, y_test, threshold=0.30)
results.append(r3)


--------------------------------------------------
  EXPERIMENT: v3 - weight {0:1,1:4}, threshold=0.30
  Threshold used: 0.3
--------------------------------------------------
  Accuracy:      0.6200
  ROC-AUC:       0.5647
  F1 (No-Show):  0.2919
  Confusion Matrix:
    TN=325  FP=137
    FN=91  TP=47

              precision    recall  f1-score   support

        Show       0.78      0.70      0.74       462
     No-Show       0.26      0.34      0.29       138

    accuracy                           0.62       600
   macro avg       0.52      0.52      0.52       600
weighted avg       0.66      0.62      0.64       600



## Experiment 4 using gridsearch tuned (scoring=f1)
Automatically finds the best hyperparameters


In [6]:
param_grid = {
    'n_estimators':     [200, 300, 500],
    'max_depth':        [None, 10, 20],
    'min_samples_leaf': [1, 2, 4],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight={0: 1, 1: 4},
        random_state=42,
        n_jobs=-1,
    ),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print(f'Best params: {grid_search.best_params_}')
model_v4 = grid_search.best_estimator_

r4 = evaluate('v4 - GridSearchCV, threshold=0.30', model_v4, X_test, y_test, threshold=0.30)
results.append(r4)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best params: {'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 200}

--------------------------------------------------
  EXPERIMENT: v4 - GridSearchCV, threshold=0.30
  Threshold used: 0.3
--------------------------------------------------
  Accuracy:      0.3567
  ROC-AUC:       0.5895
  F1 (No-Show):  0.3931
  Confusion Matrix:
    TN=89  FP=373
    FN=13  TP=125

              precision    recall  f1-score   support

        Show       0.87      0.19      0.32       462
     No-Show       0.25      0.91      0.39       138

    accuracy                           0.36       600
   macro avg       0.56      0.55      0.35       600
weighted avg       0.73      0.36      0.33       600



# Best params
### {'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 200} 

In [10]:
summary = pd.DataFrame(results)
summary = summary.set_index('name')
summary = summary.round(4)
print('\n=== EXPERIMENT SUMMARY ===')
print(summary.to_string())

best = summary['f1'].idxmax()
print(f'\nBest model by F1: {best}')


=== EXPERIMENT SUMMARY ===
                                       accuracy      f1  roc_auc  threshold
name                                                                       
v1 - balanced, threshold=0.5             0.7617  0.0892   0.5735        0.5
v2 - balanced, threshold=0.30            0.6283  0.3180   0.5735        0.3
v3 - weight {0:1,1:4}, threshold=0.30    0.6200  0.2919   0.5647        0.3
v4 - GridSearchCV, threshold=0.30        0.3567  0.3931   0.5895        0.3

Best model by F1: v4 - GridSearchCV, threshold=0.30


# Why the model kept failing?

The model is trying to learn who the risky customers are, but at the end of the day it's still a coin flip, so even the riskiest customer usually shows up anyway. The model sees "risky person -> showed up" over and over and gets confused. The fix is to stop flipping the coin and just say: if someone is risky enough, they are a no-show no randomness involved.

### If I would thhink of another example to give, it will be that 

before -> the model(the boy) is taking a test and it knew that the answer is *usualy C* on a test
now -> the model (the boy) when talking a test picks up the best answer based on the question